# PyTorch in Practice — Industry Patterns

Lessons 01-06 built every layer from scratch to build intuition.
This lesson shows how practitioners **actually use PyTorch** — leaning on
the high-level API, reusing built-in components, and following patterns
you'll encounter in real codebases and papers.

**What you'll see:**
1. The canonical train / validate / test loop template
2. Custom `Dataset` and `DataLoader`
3. Feedforward net on tabular data (`sklearn`-style)
4. CNN on images using `torchvision`
5. RNN / LSTM on time-series with the PyTorch API
6. Transformer for classification using `nn.TransformerEncoder`
7. When to use which architecture — a decision guide

> 📖 Bookmarks to keep open:
> - [PyTorch docs — nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)
> - [Karpathy — A Recipe for Training Neural Networks](http://karpathy.github.io/2019/04/25/recipe/)
> - [PyTorch tutorials index](https://pytorch.org/tutorials/)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available()          else
    "cpu"
)
print("device:", device)
print("torch:", torch.__version__)


---
## Part 1 — The Canonical Training Template

Every PyTorch training script looks roughly the same.
Learn this template once and you can read any open-source PyTorch project.


## Key Concept: The PyTorch Training Loop — Explained

Every model in this curriculum is trained with the same five-step cycle.
Here is what each step actually does and *why* it must happen in this order.

---

### The big picture: what training is

Training means repeatedly:
1. Run data through the model to get predictions
2. Measure how wrong those predictions are (the loss)
3. Figure out which weights caused the error (gradients)
4. Adjust those weights to reduce the error

PyTorch handles steps 3 and 4 automatically through **autograd** — but you
have to call the right functions in the right order.

---

### `model.train()`

Puts the model into *training mode*. This affects two layer types:

| Layer | `model.train()` | `model.eval()` |
|-------|-----------------|----------------|
| `nn.Dropout(p)` | Randomly zeros `p` fraction of neurons each forward pass | Passes all activations through unchanged |
| `nn.BatchNorm` | Computes mean/variance from the **current batch** | Uses **running statistics** accumulated during training |

For models that have neither (like the XOR net in Lesson 02), `train()` and
`eval()` make no numerical difference — but calling them is standard practice
because a collaborator might add Dropout later.

---

### `for Xb, yb in loader`

The `DataLoader` does three things you'd otherwise write yourself:
- **Batches** the dataset into chunks of `batch_size` samples
- **Shuffles** the order each epoch (if `shuffle=True`) so the model doesn't
  memorise the presentation order
- **Loads data in parallel** using background worker processes (set with `num_workers`)

Each iteration yields `Xb` (inputs) and `yb` (targets), both already converted
to tensors. Their shapes are `(batch_size, ...)`.

---

### `Xb, yb = Xb.to(device), yb.to(device)`

Copies the batch from CPU memory to the device where the model lives
(CUDA GPU or Apple MPS). Every operand in a PyTorch operation must be on
the **same** device — mixing them raises `RuntimeError: Expected all tensors
to be on the same device`.

The DataLoader always delivers tensors on CPU. Moving batches one at a time
(rather than the whole dataset) keeps memory usage bounded.

---

### `optimizer.zero_grad()`

Resets the `.grad` attribute of every model parameter to zero.

**Why is this necessary?** PyTorch *accumulates* gradients — every call to
`.backward()` **adds** to the existing `.grad` value rather than replacing it.
Skip `zero_grad` and the gradient used for this batch will be contaminated by
the gradients from every previous batch.

This accumulation is actually a deliberate design choice: it enables
*gradient accumulation*, where you run several small batches and only call
`optimizer.step()` once, simulating a larger effective batch size without
the memory cost.

```python
# What happens without zero_grad:
# Batch 1:  param.grad = g1
# Batch 2:  param.grad = g1 + g2   ← stale!
# Batch 3:  param.grad = g1 + g2 + g3  ← very stale!
```

---

### `out = model(Xb)` — the forward pass

Calling `model(Xb)` invokes `model.__call__()`, which runs your `forward()`
method and also handles hooks. As each operation runs, PyTorch builds a
**computational graph** — a record of every mathematical operation performed on
every tensor that has `requires_grad=True`.

```
Xb → Embedding → Linear → ReLU → Linear → out
         ↑           ↑       ↑       ↑
   each arrow records: "this output came from this input via this operation"
```

This graph is what makes automatic differentiation possible. It is built fresh
on every forward pass and discarded after `backward()`.

---

### `loss = criterion(out, yb)` — computing the loss

The criterion (loss function) compares `out` to `yb` and returns a scalar
tensor. Common choices:

| Task | Criterion | Input shapes |
|------|-----------|--------------|
| Binary classification | `nn.BCELoss()` | out: `(B,)`, yb: `(B,)` |
| Multi-class classification | `nn.CrossEntropyLoss()` | out: `(B, C)`, yb: `(B,)` |
| Regression | `nn.MSELoss()` | out: `(B, 1)`, yb: `(B, 1)` |

`loss` is just another tensor — it is part of the same computational graph
that started at `Xb`. The graph now records: "loss came from out, which came
from model(Xb)".

---

### `loss.backward()` — computing gradients

Traverses the computational graph **backwards** from `loss` to every
parameter, applying the chain rule at each node:

```
∂loss/∂W_last ← ∂loss/∂out × ∂out/∂W_last
∂loss/∂W_prev ← ∂loss/∂out × ∂out/∂h × ∂h/∂W_prev
...
```

Each parameter's gradient is stored in `param.grad`. After `backward()`,
the computational graph is freed from memory (PyTorch does this automatically
to avoid keeping the entire training history in RAM).

---

### `optimizer.step()` — updating the weights

Reads `param.grad` for every parameter and applies the update rule.

**SGD:**
```
param ← param - lr × param.grad
```

**Adam** (the most common choice):
```
m ← β₁ × m + (1 - β₁) × grad          # first moment (like momentum)
v ← β₂ × v + (1 - β₂) × grad²         # second moment (gradient variance)
param ← param - lr × m / (√v + ε)      # per-parameter adaptive learning rate
```
Adam adapts the learning rate for each weight individually, which is why it
converges faster than SGD on most tasks out of the box.

---

### `loss.item()`

Extracts a plain Python `float` from the loss tensor.

If you stored `loss` directly in a list, Python would keep the entire
computational graph (model activations, intermediate tensors) alive in
memory for every epoch. `.item()` returns a plain number, breaking the
reference to the graph.

---

### Why this order?

```
zero_grad → forward → loss → backward → step
```

It must be this order because:
- `zero_grad` must come before `backward` (otherwise gradients accumulate)
- `backward` must come before `step` (step reads `.grad`, which backward fills)
- `forward` must come before `loss` (you need predictions to compute error)

> 📖 [PyTorch — Autograd mechanics](https://pytorch.org/docs/stable/notes/autograd.html)
> 📖 [PyTorch — Optimizers](https://pytorch.org/docs/stable/optim.html)


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()          # enable Dropout / use batch-stats for BatchNorm
    total_loss = 0.0

    for Xb, yb in loader:
        # Move this batch from CPU RAM to the device (GPU/MPS/CPU)
        # Model weights live on `device`; data must match or PyTorch errors out
        Xb, yb = Xb.to(device), yb.to(device)

        # PyTorch ACCUMULATES gradients (adds to param.grad instead of replacing).
        # Reset before each batch so this batch's gradient doesn't mix with the last.
        optimizer.zero_grad()

        # Run data through the model. PyTorch silently records every operation on
        # tensors with requires_grad=True into a computational graph.
        out = model(Xb)

        # Compute a scalar measure of error. `loss` is the final node of the graph.
        loss = criterion(out, yb)

        # Walk the graph backwards, applying the chain rule at each node.
        # Fills param.grad for every parameter. Frees the graph from memory.
        loss.backward()

        # Read param.grad for each weight and apply the update rule.
        # SGD: param -= lr * grad   |   Adam: adaptive per-parameter lr
        optimizer.step()

        # .item() extracts a plain Python float, breaking the reference to the graph
        # so Python's GC can free the activations from this batch.
        total_loss += loss.item() * len(Xb)

    # Return mean loss per sample (not per batch), so it's comparable across
    # different batch sizes.
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion):
    model.eval()           # disable Dropout; use running stats for BatchNorm
    total_loss, correct = 0.0, 0

    # torch.no_grad(): tells autograd not to build a graph during the forward pass.
    # No backward will be called, so the graph would just waste memory.
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            out = model(Xb)
            total_loss += criterion(out, yb).item() * len(Xb)
            if out.dim() > 1 and out.shape[-1] > 1:    # multi-class
                correct += (out.argmax(1) == yb).sum().item()
            elif out.dim() == 1 or out.shape[-1] == 1:  # binary / regression
                preds = out.squeeze()
                if preds.dim() == 0:
                    preds = preds.unsqueeze(0)
                correct += ((preds > 0.5) == yb.squeeze().float()).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, criterion, optimizer,
        epochs=20, scheduler=None, patience=5):
    history = defaultdict(list)
    best_val, wait = float('inf'), 0

    for epoch in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        # ReduceLROnPlateau watches a metric and halves lr if it stops improving
        if scheduler:
            scheduler.step(vl_loss)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d} | train {tr_loss:.4f} | "
                  f"val {vl_loss:.4f} | acc {vl_acc:.2%}")

        # Early stopping: save the best weights; stop if val_loss has not
        # improved for `patience` consecutive epochs.
        if vl_loss < best_val:
            best_val, wait = vl_loss, 0
            torch.save(model.state_dict(), '/tmp/best_model.pt')
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stop at epoch {epoch}")
                break

    # Reload the best checkpoint — the final epoch may have overfit
    model.load_state_dict(torch.load('/tmp/best_model.pt', weights_only=True))
    return history


def plot_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
    ax1.plot(history['train_loss'], label='train')
    ax1.plot(history['val_loss'],   label='val')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(history['val_acc'])
    ax2.set_title('Val accuracy'); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

print("Training utilities defined.")


---
## Part 2 — Custom `Dataset`

`TensorDataset` works when your data already fits in memory as tensors.
For larger datasets (images on disk, text files, etc.) subclass `Dataset`.


In [ ]:
class SineDataset(Dataset):
    """
    Toy dataset: predict the next value in a sine wave.
    Demonstrates the three methods every Dataset must implement.
    """
    def __init__(self, n=1000, seq_len=20):
        t    = np.linspace(0, 8 * np.pi, n + seq_len)
        data = np.sin(t).astype(np.float32)
        self.X = torch.tensor(
            np.array([data[i:i+seq_len]   for i in range(n)])
        ).unsqueeze(-1)                  # (N, seq_len, 1)
        self.y = torch.tensor(
            np.array([data[i+seq_len]     for i in range(n)])
        ).unsqueeze(-1)                  # (N, 1)

    def __len__(self):
        return len(self.X)              # required — number of samples

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx] # required — single sample

dataset = SineDataset(n=800, seq_len=20)
n_val   = int(0.2 * len(dataset))
train_ds, val_ds = random_split(dataset, [len(dataset) - n_val, n_val])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)

Xb, yb = next(iter(train_loader))
print("batch X:", Xb.shape, "  y:", yb.shape)


---
## Part 3 — Feedforward Net on Tabular Data

`nn.Sequential` is the fastest way to define a straightforward stack
of layers. Use it whenever there are no branches or skip connections.

**Task:** Learn the sine wave mapping `f(t) = sin(t)` from scalar inputs.


In [ ]:
# ── Build a small feedforward net using nn.Sequential ──────────────────────
ffn = nn.Sequential(
    nn.Linear(1,  64),  nn.ReLU(),
    nn.Linear(64, 64),  nn.ReLU(),
    nn.Linear(64,  1),
).to(device)

# Scalar regression: MSE loss, Adam optimizer
criterion_ff = nn.MSELoss()
optimizer_ff = optim.Adam(ffn.parameters(), lr=1e-3)
scheduler_ff = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_ff, patience=3, factor=0.5
)

# Simple scalar dataset (not a sequence)
t_train = torch.linspace(0, 6*np.pi, 500).unsqueeze(1).to(device)
y_train = torch.sin(t_train)
t_val   = torch.linspace(6*np.pi, 8*np.pi, 100).unsqueeze(1).to(device)
y_val   = torch.sin(t_val)

ff_train_ds = TensorDataset(t_train, y_train)
ff_val_ds   = TensorDataset(t_val,   y_val)
ff_tr_loader = DataLoader(ff_train_ds, batch_size=32, shuffle=True)
ff_vl_loader = DataLoader(ff_val_ds,   batch_size=32, shuffle=False)

h = fit(ffn, ff_tr_loader, ff_vl_loader,
        criterion_ff, optimizer_ff, epochs=100,
        scheduler=scheduler_ff, patience=10)

# Plot predictions
ffn.eval()
with torch.no_grad():
    t_plot = torch.linspace(0, 8*np.pi, 400).unsqueeze(1).to(device)
    preds  = ffn(t_plot).cpu().squeeze().numpy()
plt.figure(figsize=(9, 3))
plt.plot(t_plot.cpu(), np.sin(t_plot.cpu()), label='true sin(t)', alpha=0.7)
plt.plot(t_plot.cpu(), preds, '--', label='FFN prediction', alpha=0.9)
plt.axvline(6*np.pi, color='gray', linestyle=':', label='train / val split')
plt.legend(); plt.title('Feedforward net — sine regression'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


---
## Part 4 — 1-D CNN on a Sequence

Use `nn.Conv1d` for any task where **local patterns** matter most —
keyword spotting, time-series anomaly detection, short-text classification.
The network here classifies whether a sine-wave segment is in the *first*
or *second* half of a period (a proxy for phase detection).

> 📖 [PyTorch — nn.Conv1d](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html)


In [ ]:
# ── CNN for binary classification on 1-D sequences ─────────────────────────
class CNN1D(nn.Module):
    def __init__(self, in_channels=1, seq_len=20):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(32,          64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),   # global average pooling → (B, 64, 1)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),              # (B, 64)
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (B, seq_len, 1)  →  need (B, 1, seq_len) for Conv1d
        return self.fc(self.conv(x.permute(0, 2, 1))).squeeze(-1)

# Binary label: sin rising (1) vs falling (0) — first derivative sign
def make_cnn_dataset(n=800, seq_len=20):
    t    = np.linspace(0, 20*np.pi, n + seq_len + 1).astype(np.float32)
    data = np.sin(t)
    Xs, ys = [], []
    for i in range(n):
        seg   = data[i:i+seq_len]
        label = 1 if data[i+seq_len] > data[i+seq_len-1] else 0
        Xs.append(seg); ys.append(label)
    X = torch.tensor(np.array(Xs)).unsqueeze(-1)   # (N, seq_len, 1)
    y = torch.tensor(np.array(ys, dtype=np.float32))
    return TensorDataset(X, y)

cnn_ds = make_cnn_dataset()
cnn_tr, cnn_vl = random_split(cnn_ds, [640, 160])
cnn_tr_loader = DataLoader(cnn_tr, batch_size=32, shuffle=True)
cnn_vl_loader = DataLoader(cnn_vl, batch_size=32, shuffle=False)

cnn_model = CNN1D().to(device)
h_cnn = fit(cnn_model, cnn_tr_loader, cnn_vl_loader,
            nn.BCELoss(),
            optim.Adam(cnn_model.parameters(), lr=1e-3),
            epochs=30, patience=8)
plot_history(h_cnn)


---
## Part 5 — LSTM via the PyTorch API

`nn.LSTM` handles everything: multi-layer, dropout between layers,
bidirectional processing. You rarely need to implement an LSTM cell manually.

Key returns:
```
output, (h_n, c_n) = lstm(x)
#  Shapes (B = batch_size, L = seq_len):
#  output: (B, L, hidden)  — hidden state at every position in the sequence
#  h_n:    (num_layers, B, hidden)  — final hidden state (last position only)
#  c_n:    (num_layers, B, hidden)  — final cell state   (long-term memory)
```


In [ ]:
class LSTMRegressor(nn.Module):
    """Drop-in LSTM for sequence regression.  Industry-standard structure."""
    def __init__(self, input_size=1, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden,
            num_layers=layers,
            dropout=dropout,         # applied between layers (not on last)
            batch_first=True,        # convention: (B, L, features)
        )
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)        # out: (B, L, hidden)
        return self.fc(out[:, -1])   # last timestep → (B, 1)

lstm_model = LSTMRegressor().to(device)

# Reuse the SineDataset from Part 2
h_lstm = fit(lstm_model, train_loader, val_loader,
             nn.MSELoss(),
             optim.Adam(lstm_model.parameters(), lr=1e-3),
             epochs=50, patience=10)
plot_history(h_lstm)


---
## Part 6 — Transformer via `nn.TransformerEncoder`

PyTorch ships a production-ready `nn.TransformerEncoder`. You don't need to
implement attention yourself for classification tasks.

For most production use cases you'd fine-tune a pre-trained model
(BERT, RoBERTa, GPT-2 via HuggingFace) rather than train from scratch.
This section shows the native PyTorch path so you understand what
HuggingFace wrappers are actually doing under the hood.

> 📖 [PyTorch — nn.TransformerEncoder](https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html)
> 📖 [HuggingFace Transformers](https://huggingface.co/docs/transformers/index)


In [ ]:
import math

class TransformerClassifier(nn.Module):
    """
    Encoder-only Transformer for sequence classification.
    Uses PyTorch's built-in nn.TransformerEncoder — no custom attention.
    """
    def __init__(self, input_size=1, d_model=64, nhead=4,
                 num_layers=2, dim_ff=128, dropout=0.1, num_classes=1):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        encoder_layer   = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,          # (B, L, d_model)
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_head = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(),
            nn.Linear(32, num_classes),
            nn.Sigmoid() if num_classes == 1 else nn.Identity(),
        )

    def forward(self, x):
        x = self.input_proj(x)        # (B, L, d_model)
        x = self.encoder(x)           # self-attention across all timesteps
        x = x.mean(dim=1)             # mean-pool over sequence → (B, d_model)
        return self.cls_head(x).squeeze(-1)

tf_model = TransformerClassifier().to(device)

# Reuse CNN binary-classification dataset (rising vs falling sine)
h_tf = fit(tf_model, cnn_tr_loader, cnn_vl_loader,
           nn.BCELoss(),
           optim.Adam(tf_model.parameters(), lr=1e-3, weight_decay=1e-4),
           epochs=40, patience=10)
plot_history(h_tf)


---
## Part 7 — Saving and Loading Models

Two patterns:

| Pattern | `torch.save(...)` target | Use case |
|---------|--------------------------|----------|
| **Weights only** | `model.state_dict()` | Resuming in same codebase |
| **Full checkpoint** | `{'state': ..., 'opt': ..., 'epoch': ...}` | Resuming training after a crash |

Prefer **weights-only** saves unless you need to resume mid-training.


In [ ]:
# ── Save / load weights only ────────────────────────────────────────────────
torch.save(lstm_model.state_dict(), '/tmp/lstm_weights.pt')

# To reload:
lstm_reload = LSTMRegressor().to(device)
lstm_reload.load_state_dict(
    torch.load('/tmp/lstm_weights.pt', weights_only=True)
)
lstm_reload.eval()
print("Loaded model:", lstm_reload)

# ── Full checkpoint (for resuming training) ─────────────────────────────────
checkpoint = {
    'epoch':      50,
    'model':      lstm_model.state_dict(),
    'optimizer':  optim.Adam(lstm_model.parameters()).state_dict(),
    'val_loss':   min(h_lstm['val_loss']),
}
torch.save(checkpoint, '/tmp/checkpoint.pt')

ckpt = torch.load('/tmp/checkpoint.pt', weights_only=False)
print(f"Saved at epoch {ckpt['epoch']}, best val loss {ckpt['val_loss']:.5f}")


---
## Part 8 — When to Use Which Architecture

| Task | Data | First choice | Why |
|------|------|--------------|-----|
| Tabular / structured | Rows of numbers | **Feedforward** | No spatial or temporal structure |
| Short text / document classification | Text <512 tokens | **Fine-tune BERT** | Pretrained language understanding |
| Image classification | Grid of pixels | **CNN** (or ViT for large budgets) | Local spatial patterns |
| Time-series forecast | 1-D sequence | **LSTM** | Long-range dependencies, fast to train |
| Long-sequence NLP | Text >512 tokens | **Transformer** | Parallelises; scales better than LSTM |
| Speech / audio | Waveform | **1-D CNN → LSTM** or **Wav2Vec** | Local + temporal hierarchy |
| Generative / autoregressive | Any sequence | **Transformer decoder** (GPT-style) | State of the art for generation |

### Red flags

- **Don't train a Transformer from scratch on <100k samples.** Pretrain or use LSTM.
- **Don't use an RNN if sequence length > 500.** Vanishing gradients despite LSTM.
  Use a Transformer or chunked approach.
- **Don't skip gradient clipping with RNNs.** `clip_grad_norm_(params, 1.0)` is
  near-mandatory.

> 📖 [PyTorch Model Hub](https://pytorch.org/hub/) — pretrained models for
> vision, NLP, and audio you can fine-tune in a few lines.


---
### Concept Check: PyTorch Patterns

**Q1.** In `train_one_epoch`, `optimizer.zero_grad()` is called *inside* the
loop (once per batch). What would happen if you moved it *outside* the loop
and called it just once before the loop started?

**Q2.** `model.train()` and `model.eval()` affect Dropout and BatchNorm.
In a plain LSTM with no Dropout, does toggling between these modes change
the model's outputs? Why do we still call them?

**Q3.** The `fit()` function saves the best model to disk and reloads it at
the end. Why reload the best checkpoint rather than using the weights from
the final epoch?

**Q4.** A colleague says "LSTMs are obsolete — just use Transformers for
everything." Give one scenario where you'd still choose an LSTM in 2024.


In [ ]:
# Q1:
# Q2:
# Q3:
# Q4: